# Bilateral Investment Treaty Panel Development

This notebook reconstructs annual bilateral investment treaty relationships with China and the United States for the economic structural-ties component of the US-China Closeness Index.

The analytical state is evaluated at 31 December of each year, consistent with the other annual project components.

## Institutional-state rules

For each treaty and year-end:

- `not_yet_signed`: signature occurs after year-end;
- `signed_not_in_force`: signed by year-end but not yet effective;
- `in_force`: effective by year-end and not terminated by year-end;
- `terminated`: termination occurs on or before year-end.

At the country-anchor-year level, any treaty in force controls the current state. A signed-only treaty controls only when no treaty is already in force. Terminated agreements receive no current score but remain visible as historical institutional ties.

## BIT structural score

The provisional BIT score is:

```text
none or historical termination = 0.00
signed but not in force        = 0.25
in force                       = 1.00
```

The partial score for signed-only agreements recognizes completed negotiation and institutional intent without equating signature with an operational treaty framework.

This BIT score will later become one subcomponent of economic institutional ties. It is not the complete economic score.

In [1]:
# locate repository and define paths
from pathlib import Path
import subprocess
import sys

import pandas as pd


def locate_repository_root(start=None):
    try:
        start_path = Path(start or Path.cwd()).resolve()
    except FileNotFoundError:
        start_path = Path.home()

    candidates = [start_path]
    candidates.extend(start_path.parents)

    for candidate in candidates:
        if (
            (candidate / "src" / "data").exists()
            and (candidate / "data").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate

    raise FileNotFoundError("Could not locate the repository root.")


REPOSITORY_ROOT = locate_repository_root()
SCRIPT = REPOSITORY_ROOT / "src" / "data" / "build_bilateral_investment_treaties.py"
OUTPUT = REPOSITORY_ROOT / "data" / "processed" / "bilateral_investment_treaties_country_anchor_year.csv"
REPORTS = REPOSITORY_ROOT / "outputs" / "tables" / "bilateral_investment_treaties"
CHINA_SOURCE = REPOSITORY_ROOT / "data" / "sources" / "relations" / "Bilateral Investment Treaty with China.xlsx"
US_SOURCE = REPOSITORY_ROOT / "data" / "sources" / "relations" / "Bilateral Investment Treaty with US.xlsx"

print("Repository root located.")
print("Builder exists:", SCRIPT.exists())
print("China BIT workbook exists:", CHINA_SOURCE.exists())
print("US BIT workbook exists:", US_SOURCE.exists())

Repository root located.
Builder exists: True
China BIT workbook exists: True
US BIT workbook exists: True


In [2]:
# run the BIT builder
completed = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=REPOSITORY_ROOT,
    check=False,
    text=True,
    capture_output=True,
)

if completed.stdout.strip():
    print(completed.stdout)

if completed.stderr.strip():
    print("SCRIPT STDERR:")
    print(completed.stderr)

if completed.returncode != 0:
    raise RuntimeError(
        f"{SCRIPT.name} failed with exit code {completed.returncode}."
    )

BILATERAL INVESTMENT TREATY PANEL
{
  "rows": 9612,
  "countries_or_historical_entities": 199,
  "years": [
    2000,
    2024
  ],
  "state_counts": {
    "none": 5345,
    "in_force": 3554,
    "signed_not_in_force": 637,
    "historical_terminated": 76
  },
  "anchor_state_counts": {
    "China|historical_terminated": 56,
    "China|in_force": 2581,
    "China|none": 1701,
    "China|signed_not_in_force": 468,
    "United States|historical_terminated": 20,
    "United States|in_force": 973,
    "United States|none": 3644,
    "United States|signed_not_in_force": 169
  },
  "validation_failures": {
    "duplicate_keys": 0,
    "blank_country_names": 0,
    "raw_score_out_of_bounds": 0,
    "scaled_score_out_of_bounds": 0,
    "in_force_score_mismatch": 0,
    "signed_score_mismatch": 0
  },
  "treaty_records": {
    "China": 150,
    "United States": 47,
    "expanded_country_treaty_records": 199
  }
}

Production output: data/processed/bilateral_investment_treaties_country_anchor_ye

In [3]:
# load production and audit outputs
panel = pd.read_csv(OUTPUT, low_memory=False)
register = pd.read_csv(
    REPORTS / "bit_treaty_register_normalized.csv",
    low_memory=False,
)
events = pd.read_csv(
    REPORTS / "bit_treaty_year_states.csv",
    low_memory=False,
)
crosswalk = pd.read_csv(
    REPORTS / "bit_counterparty_crosswalk.csv",
    low_memory=False,
)

print("Production rows:", len(panel))
print("Countries or historical entities:", panel["country_iso3"].nunique())
print("Normalized treaty-country records:", len(register))
print("Treaty-year audit rows:", len(events))

Production rows: 9612
Countries or historical entities: 199
Normalized treaty-country records: 199
Treaty-year audit rows: 4975


In [4]:
# validate keys, bounds and state-score agreement
keys = ["country_iso3", "anchor", "year"]

assert len(panel) == 9612
assert panel["country_iso3"].nunique() == 199
assert not panel.duplicated(keys).any()
assert panel["country_name"].notna().all()
assert panel["bit_structural_score_raw"].between(0, 1).all()
assert panel["bit_structural_score_scaled"].between(0, 100).all()

assert panel.loc[
    panel["bit_state_year_end"].eq("in_force"),
    "bit_structural_score_raw",
].eq(1).all()

assert panel.loc[
    panel["bit_state_year_end"].eq("signed_not_in_force"),
    "bit_structural_score_raw",
].eq(0.25).all()

assert panel.loc[
    panel["bit_state_year_end"].isin(["none", "historical_terminated"]),
    "bit_structural_score_raw",
].eq(0).all()

print("PASS: production keys are unique")
print("PASS: scores lie within valid bounds")
print("PASS: year-end states and scores agree")

PASS: production keys are unique
PASS: scores lie within valid bounds
PASS: year-end states and scores agree


In [5]:
# inspect source records and current states
source_counts = (
    register.groupby(["anchor", "Status"])
    .size()
    .rename("treaty_country_records")
    .reset_index()
)

state_counts = (
    panel.groupby(["anchor", "bit_state_year_end"])
    .size()
    .rename("country_anchor_years")
    .reset_index()
)

display(source_counts)
display(state_counts)

,anchor,Status,treaty_country_records
0,China,In force,110
1,China,Signed,17
2,China,Terminated,25
3,United States,In force,39
4,United States,Signed,6
5,United States,Terminated,2


,anchor,bit_state_year_end,country_anchor_years
0,China,historical_terminated,56
1,China,in_force,2581
2,China,none,1701
3,China,signed_not_in_force,468
4,United States,historical_terminated,20
5,United States,in_force,973
6,United States,none,3644
7,United States,signed_not_in_force,169


## Source normalization note

The China workbook contains 150 treaty records and the United States workbook contains 47 treaty records. The Belgium-Luxembourg Economic Union treaty is expanded to both Belgium and Luxembourg, producing 199 treaty-country records after normalization.

The source workbooks remain the treaty-level authority. The annual panel is a reproducible year-end reconstruction.

In [6]:
# validate counterparty resolution
assert crosswalk["country_iso3"].notna().all()
assert not crosswalk.duplicated(
    ["anchor", "counterparty_source_name", "country_iso3"]
).any()

multi_country = crosswalk[
    crosswalk["multi_country_source_entity"]
]

print("PASS: every treaty counterparty resolves to a country identifier")
print("Multi-country source entities:", len(multi_country))
display(multi_country)

PASS: every treaty counterparty resolves to a country identifier
Multi-country source entities: 2


,counterparty_source_name,counterparty_normalized_name,country_iso3,anchor,multi_country_source_entity
8,BLEU (Belgium-Luxembourg Economic Union),bleu belgium luxembourg economic union,BEL,China,True
74,BLEU (Belgium-Luxembourg Economic Union),bleu belgium luxembourg economic union,LUX,China,True


In [7]:
# inspect year-end transitions and replacements
transitions = panel.sort_values(keys).copy()
transitions["prior_state"] = (
    transitions.groupby(["country_iso3", "anchor"])["bit_state_year_end"]
    .shift(1)
)
transitions = transitions[
    transitions["prior_state"].notna()
    & transitions["prior_state"].ne(transitions["bit_state_year_end"])
]

transition_counts = (
    transitions.groupby(["anchor", "prior_state", "bit_state_year_end"])
    .size()
    .rename("transitions")
    .reset_index()
)

replacement_overlap = panel[panel["bit_replacement_overlap"]].copy()

display(transition_counts)
print("Country-anchor-years with multiple treaties in force:", len(replacement_overlap))
display(
    replacement_overlap[
        [
            "country_iso3",
            "country_name",
            "anchor",
            "year",
            "bit_in_force_count",
            "bit_current_treaty_titles",
        ]
    ].head(30)
)

,anchor,prior_state,bit_state_year_end,transitions
0,China,in_force,historical_terminated,7
1,China,none,in_force,3
2,China,none,signed_not_in_force,32
3,China,signed_not_in_force,in_force,29
4,United States,in_force,historical_terminated,2
5,United States,none,signed_not_in_force,2
6,United States,signed_not_in_force,in_force,10


Country-anchor-years with multiple treaties in force: 0


,country_iso3,country_name,anchor,year,bit_in_force_count,bit_current_treaty_titles


## Replacement continuity

The annual state aggregates across treaty versions. If an old treaty terminates and a replacement is already in force at the same year-end, the country-anchor relationship remains `in_force`.

Treaty IDs and titles remain available for audit, while the structural score reflects the operating country-anchor framework rather than the number of treaty documents.

In [8]:
# compare China and United States BIT ties in the latest year
latest_year = int(panel["year"].max())
latest = panel[panel["year"].eq(latest_year)]

latest_states = (
    latest.groupby(["anchor", "bit_state_year_end"])
    .size()
    .rename("countries")
    .reset_index()
)

display(latest_states)

comparison = latest.pivot(
    index=["country_iso3", "country_name"],
    columns="anchor",
    values="bit_structural_score_raw",
).reset_index()
comparison["china_minus_us_bit_score"] = (
    comparison["China"] - comparison["United States"]
)

display(comparison.nlargest(20, "china_minus_us_bit_score"))
display(comparison.nsmallest(20, "china_minus_us_bit_score"))

,anchor,bit_state_year_end,countries
0,China,historical_terminated,7
1,China,in_force,109
2,China,none,63
3,China,signed_not_in_force,17
4,United States,historical_terminated,2
5,United States,in_force,39
6,United States,none,149
7,United States,signed_not_in_force,6


anchor,country_iso3,country_name,China,United States,china_minus_us_bit_score
1,AGO,Angola,1.0,0.0,1.0
4,ARE,United Arab Emirates,1.0,0.0,1.0
8,AUS,Australia,1.0,0.0,1.0
9,AUT,Austria,1.0,0.0,1.0
12,BEL,Belgium,1.0,0.0,1.0
19,BIH,Bosnia and Herzegovina,1.0,0.0,1.0
24,BRB,Barbados,1.0,0.0,1.0
28,CAN,Canada,1.0,0.0,1.0
29,CHE,Switzerland,1.0,0.0,1.0
36,COL,Colombia,1.0,0.0,1.0


anchor,country_iso3,country_name,China,United States,china_minus_us_bit_score
68,GRD,Grenada,0.00,1.00,-1.00
72,HND,Honduras,0.00,1.00,-1.00
136,PAN,Panama,0.00,1.00,-1.00
149,RWA,Rwanda,0.00,1.00,-1.00
152,SEN,Senegal,0.00,1.00,-1.00
85,JOR,Jordan,0.25,1.00,-0.75
74,HTI,Haiti,0.00,0.25,-0.25
128,NIC,Nicaragua,0.00,0.25,-0.25
156,SLV,El Salvador,0.00,0.25,-0.25
0,AFG,Afghanistan,0.00,0.00,0.00


In [9]:
# examine signed-only and terminated-history cases
signed_only = panel[
    panel["bit_state_year_end"].eq("signed_not_in_force")
].copy()
terminated = panel[
    panel["bit_state_year_end"].eq("historical_terminated")
].copy()

print("Signed-only annual observations:", len(signed_only))
print("Historical-termination annual observations:", len(terminated))

display(
    signed_only[
        [
            "country_iso3",
            "country_name",
            "anchor",
            "year",
            "bit_signed_not_in_force_count",
            "bit_all_year_end_states",
        ]
    ].tail(30)
)

display(
    terminated[
        [
            "country_iso3",
            "country_name",
            "anchor",
            "year",
            "bit_terminated_count",
            "bit_ever_in_force",
        ]
    ].tail(30)
)

Signed-only annual observations: 637
Historical-termination annual observations: 76


,country_iso3,country_name,anchor,year,bit_signed_not_in_force_count,bit_all_year_end_states
9323,VUT,Vanuatu,China,2011,1,signed_not_in_force
9324,VUT,Vanuatu,China,2012,1,signed_not_in_force
9325,VUT,Vanuatu,China,2013,1,signed_not_in_force
9326,VUT,Vanuatu,China,2014,1,signed_not_in_force
9327,VUT,Vanuatu,China,2015,1,signed_not_in_force
9328,VUT,Vanuatu,China,2016,1,signed_not_in_force
9329,VUT,Vanuatu,China,2017,1,signed_not_in_force
9330,VUT,Vanuatu,China,2018,1,signed_not_in_force
9331,VUT,Vanuatu,China,2019,1,signed_not_in_force
9332,VUT,Vanuatu,China,2020,1,signed_not_in_force


,country_iso3,country_name,anchor,year,bit_terminated_count,bit_ever_in_force
2474,ECU,Ecuador,United States,2022,1,True
2475,ECU,Ecuador,United States,2023,1,True
2476,ECU,Ecuador,United States,2024,1,True
3752,IDN,Indonesia,China,2015,1,True
3753,IDN,Indonesia,China,2016,1,True
3754,IDN,Indonesia,China,2017,1,True
3755,IDN,Indonesia,China,2018,1,True
3756,IDN,Indonesia,China,2019,1,True
3757,IDN,Indonesia,China,2020,1,True
3758,IDN,Indonesia,China,2021,1,True


## Production conclusion

The BIT panel now provides a symmetric country-anchor-year structure with:

- treaty-level provenance;
- year-end institutional states;
- active, signed-only, and historical-termination flags;
- replacement continuity;
- treaty counts and identifiers; and
- a provisional 0, 0.25, or 1 structural score.

BITs remain one economic institutional subcomponent. Free-trade agreements and regional economic frameworks will be developed separately before the economic structural-ties score is assembled.

# Decision register

## Decided

- Use institutional state at 31 December of each year.
- Give an in-force BIT a score of 1.
- Give a signed but not yet effective BIT a provisional score of 0.25.
- Give no current score to a terminated treaty while preserving its history.
- Aggregate treaty versions at country-anchor-year level.
- Preserve treaty-level records and annual states for audit.
- Keep BITs inside economic structural ties, separate from security and political treaties.

## Next

- Review BIT edge cases and replacement overlaps.
- Compare reconstructed BIT states with the pre-coded treaty workbook.
- Build bilateral and regional trade-agreement subcomponents.
- Assemble the wider economic institutional-ties score.
- Integrate trade intensity and other economic measures later.